<a href="https://colab.research.google.com/github/rymadinari/-arene-des-algos-Ryma-Dinari-/blob/main/Jour_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# PHASE A : Régression — California Housing
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np
import pandas as pd

def charger_immobilier():
    data = fetch_california_housing()
    X, y = data.data, data.target
    print(f"California Housing : {X.shape}, cible = prix médian en centaines de milliers de $")
    print(f"Variables : {data.feature_names}")
    return X, y

def evaluer_regression(modele, X_train, X_test, y_train, y_test):
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)
    return {
        "r2"  : r2_score(y_test, y_pred),
        "mae" : mean_absolute_error(y_test, y_pred),
        "rmse": root_mean_squared_error(y_test, y_pred),
    }

X, y = charger_immobilier()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

for nom, modele in [
    ("LinearRegression", LinearRegression()),
    ("RandomForest",     RandomForestRegressor(n_estimators=100, random_state=42)),
]:
    scores = evaluer_regression(modele, X_train_s, X_test_s, y_train, y_test)
    print(f"{nom:<20} R2={scores['r2']:.2f}  MAE={scores['mae']:.2f}  RMSE={scores['rmse']:.2f}")

California Housing : (20640, 8), cible = prix médian en centaines de milliers de $
Variables : ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
LinearRegression     R2=0.58  MAE=0.53  RMSE=0.75
RandomForest         R2=0.81  MAE=0.33  RMSE=0.51


In [7]:
# CHECKPOINTS PHASE A

# Checkpoint 1 : cas normal
print("CHECKPOINT 1 — Dataset complet")
for nom, modele in [
    ("LinearRegression", LinearRegression()),
    ("RandomForest",     RandomForestRegressor(n_estimators=100, random_state=42)),
]:
    scores = evaluer_regression(modele, X_train_s, X_test_s, y_train, y_test)
    print(f"{nom:<20} R2={scores['r2']:.2f}  MAE={scores['mae']:.2f}  RMSE={scores['rmse']:.2f}")

print()

# Checkpoint 2 : cas limite (100 lignes seulement)
print("CHECKPOINT 2 — Seulement 100 lignes d'entraînement")
X_100 = X_train_s[:100]
y_100 = y_train[:100]

for nom, modele in [
    ("LinearRegression", LinearRegression()),
    ("RandomForest",     RandomForestRegressor(n_estimators=100, random_state=42)),
]:
    scores = evaluer_regression(modele, X_100, X_test_s, y_100, y_test)
    print(f"{nom:<20} R2={scores['r2']:.2f}  MAE={scores['mae']:.2f}  RMSE={scores['rmse']:.2f}")

print("""
Observation : le R2 s'effondre (peut devenir négatif sur le Random Forest).
Parceque Avec 100 lignes sur 20 640, le modèle n'a pas vu assez de cas
pour généraliser. Le Random Forest surfit : il mémorise les 100 exemples
au lieu d'apprendre une vraie règle. La régression linéaire résiste mieux
car elle a moins de paramètres à estimer.
""")
print()

# Checkpoint 3 : cas adversarial
print("CHECKPOINT 3 — Quartier fictif hors plage")

lr_final = LinearRegression()
lr_final.fit(X_train_s, y_train)


quartier_fictif = np.array([[0, 20, 5, 1, 9000, 3, 37.0, -120.0]])

quartier_fictif_s = scaler.transform(quartier_fictif)

pred_lr = lr_final.predict(quartier_fictif_s)[0]

rf_final = RandomForestRegressor(n_estimators=100, random_state=42)
rf_final.fit(X_train_s, y_train)
pred_rf = rf_final.predict(quartier_fictif_s)[0]

print(f"Quartier fictif : revenu=0, population=9000")
print(f"LinearRegression → prix prédit : {pred_lr:.2f} (x100k$)")
print(f"RandomForest     → prix prédit : {pred_rf:.2f} (x100k$)")

if pred_lr < 0:
    print(f"  Régression linéaire : prix NÉGATIF ({pred_lr:.2f}) — valeur absurde !")
else:
    print(f" Valeur hors plage des données d'entraînement — à surveiller")

print(f"Plage normale des prix dans le dataset : "
      f"{y.min():.2f} à {y.max():.2f} (x100k$)")

print("""
Que faire en production ?
1. Valider les entrées AVANT le modèle : rejeter tout revenu < 0
   ou toute population hors de la plage vue à l'entraînement.
2. Ajouter une couche de détection d'anomalies en amont.
3. Ne jamais faire confiance à une prédiction sur une entrée
   que le modèle n'a jamais vue : extrapoler, c'est risqué.
""")

CHECKPOINT 1 — Dataset complet
LinearRegression     R2=0.58  MAE=0.53  RMSE=0.75
RandomForest         R2=0.81  MAE=0.33  RMSE=0.51

CHECKPOINT 2 — Seulement 100 lignes d'entraînement
LinearRegression     R2=0.40  MAE=0.54  RMSE=0.88
RandomForest         R2=0.54  MAE=0.56  RMSE=0.77

Observation : le R2 s'effondre (peut devenir négatif sur le Random Forest).
Parceque Avec 100 lignes sur 20 640, le modèle n'a pas vu assez de cas
pour généraliser. Le Random Forest surfit : il mémorise les 100 exemples
au lieu d'apprendre une vraie règle. La régression linéaire résiste mieux
car elle a moins de paramètres à estimer.


CHECKPOINT 3 — Quartier fictif hors plage
Quartier fictif : revenu=0, population=9000
LinearRegression → prix prédit : -0.18 (x100k$)
RandomForest     → prix prédit : 0.71 (x100k$)
  Régression linéaire : prix NÉGATIF (-0.18) — valeur absurde !
Plage normale des prix dans le dataset : 0.15 à 5.00 (x100k$)

Que faire en production ?
1. Valider les entrées AVANT le modèle : rej

In [8]:
 # PHASE B : Clustering — AirBnB

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

def charger_airbnb(url_csv):
    df = pd.read_csv(url_csv)
    cols = ["price", "minimum_nights", "number_of_reviews", "availability_365"]
    df = df[cols].copy()

    if df["price"].dtype == "object":
        df["price"] = df["price"].str.replace("[$,]", "", regex=True).astype(float)

    df = df.dropna()
    print(f"Listings chargés : {df.shape[0]} lignes, {df.shape[1]} colonnes retenues")
    return df

def choisir_k(X_scaled, k_range=range(2, 9)):
    print(f"{'k':<5} {'Inertie':>10} {'Silhouette':>12}")
    print("-" * 30)
    meilleur_k, meilleur_score = 2, -1
    for k in k_range:
        km = KMeans(n_clusters=k, n_init=10, random_state=42)
        labels = km.fit_predict(X_scaled)
        inertie  = km.inertia_
        silhouette = silhouette_score(X_scaled, labels)
        print(f"{k:<5} {inertie:>10.0f} {silhouette:>12.2f}")
        if silhouette > meilleur_score:
            meilleur_score = silhouette
            meilleur_k = k
    print(f"Segment retenu : k={meilleur_k} (meilleure silhouette)")
    return meilleur_k


url = "https://data.insideairbnb.com/france/ile-de-france/paris/2024-06-10/visualisations/listings.csv"
df_airbnb = charger_airbnb(url)

scaler_ab = StandardScaler()
X_ab = scaler_ab.fit_transform(df_airbnb)

k_optimal = choisir_k(X_ab)

km_final = KMeans(n_clusters=k_optimal, n_init=10, random_state=42)
df_airbnb["segment"] = km_final.fit_predict(X_ab)
print("Description des segments :")
print(df_airbnb.groupby("segment").mean().round(1))

Listings chargés : 74579 lignes, 4 colonnes retenues
k        Inertie   Silhouette
------------------------------
2         230941         0.81
3         173481         0.44
4         133196         0.47
5          94984         0.48
6          83293         0.49
7          73579         0.50
8          63797         0.38
Segment retenu : k=2 (meilleure silhouette)
Description des segments :
         price  minimum_nights  number_of_reviews  availability_365
segment                                                            
0        290.0             6.1               22.2             157.3
1        202.5           360.4               16.5             304.7


In [9]:
#Cas limite : SANS standardiser
print("CAS LIMITE — KMeans SANS standardisation")
X_brut = df_airbnb[["price","minimum_nights","number_of_reviews","availability_365"]].values

km_brut = KMeans(n_clusters=k_optimal, n_init=10, random_state=42)
df_airbnb["segment_brut"] = km_brut.fit_predict(X_brut)

print("Taille des segments SANS scaling :")
print(df_airbnb["segment_brut"].value_counts())
print(df_airbnb.groupby("segment_brut")[["price","minimum_nights"]].mean().round(1))
print("""
Observation : la colonne 'price' (en centaines) écrase toutes les autres.
Les clusters se forment uniquement sur le prix, les autres variables
sont ignorées. Le clustering ne fait plus que trier par prix.
→ Sans standardiser, KMeans est inutile sur des colonnes d'échelles différentes.
""")


#Cas adversarial : annonce à 100 000€ la nuit
print("CAS ADVERSARIAL — Valeur aberrante 100 000€/nuit")

df_piege = df_airbnb[["price","minimum_nights","number_of_reviews","availability_365"]].copy()

outlier = pd.DataFrame([[100000, 1, 0, 365]],
                        columns=df_piege.columns)
df_piege = pd.concat([df_piege, outlier], ignore_index=True)

scaler_piege = StandardScaler()
X_piege = scaler_piege.fit_transform(df_piege)

km_piege = KMeans(n_clusters=k_optimal, n_init=10, random_state=42)
km_piege.fit(X_piege)

print("Centres des clusters AVEC l'outlier :")
print(pd.DataFrame(km_piege.cluster_centers_,
      columns=df_piege.columns).round(2))
print("""
Observation : l'outlier à 100 000€ attire un cluster entier vers lui.
Les autres segments sont déformés pour "s'éloigner" de ce point extrême.
→ Un seul outlier non nettoyé ruine toute la segmentation.
→ C'est exactement pour ça que le nettoyage J2 est un prérequis absolu.
""")

CAS LIMITE — KMeans SANS standardisation
Taille des segments SANS scaling :
segment_brut
0    74356
1      223
Name: count, dtype: int64
               price  minimum_nights
segment_brut                        
0              262.1            10.0
1             9275.9             9.0

Observation : la colonne 'price' (en centaines) écrase toutes les autres.
Les clusters se forment uniquement sur le prix, les autres variables
sont ignorées. Le clustering ne fait plus que trier par prix.
→ Sans standardiser, KMeans est inutile sur des colonnes d'échelles différentes.

CAS ADVERSARIAL — Valeur aberrante 100 000€/nuit
Centres des clusters AVEC l'outlier :
   price  minimum_nights  number_of_reviews  availability_365
0   0.00            -0.1                0.0             -0.01
1  -0.11             9.0               -0.1              1.28

Observation : l'outlier à 100 000€ attire un cluster entier vers lui.
Les autres segments sont déformés pour "s'éloigner" de ce point extrême.
→ Un seul 

In [10]:
# PHASE C : Spam (texte)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

def charger_spam():
    url = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/sms.tsv"
    df = pd.read_csv(url, sep="\t", header=None, names=["label", "message"])
    df["label"] = (df["label"] == "spam").astype(int)
    print(f"SMS chargés : {df.shape[0]} messages")
    print(f"Spam : {df['label'].sum()} | Normal : {(df['label']==0).sum()}")
    return df["message"].tolist(), df["label"].tolist()

def vectoriser_textes(messages, vectorizer=None):
    if vectorizer is None:
        vectorizer = TfidfVectorizer()
        X = vectorizer.fit_transform(messages)
    else:
        X = vectorizer.transform(messages)
    return X, vectorizer

def evaluer_spam(modele, X_train, X_test, y_train, y_test):
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)
    print(classification_report(y_test, y_pred, target_names=["normal", "spam"]))

messages, labels = charger_spam()

msg_train, msg_test, y_train, y_test = train_test_split(
    messages, labels, test_size=0.2, random_state=42, stratify=labels
)

X_train_txt, vect = vectoriser_textes(msg_train)
X_test_txt,  _    = vectoriser_textes(msg_test, vectorizer=vect)

for nom, modele in [
    ("Naive Bayes",           MultinomialNB()),
    ("Logistic Regression",   LogisticRegression(max_iter=1000)),
]:
    print(f"\n=== {nom} ===")
    evaluer_spam(modele, X_train_txt, X_test_txt, y_train, y_test)

SMS chargés : 5572 messages
Spam : 747 | Normal : 4825

=== Naive Bayes ===
              precision    recall  f1-score   support

      normal       0.96      1.00      0.98       966
        spam       1.00      0.70      0.83       149

    accuracy                           0.96      1115
   macro avg       0.98      0.85      0.90      1115
weighted avg       0.96      0.96      0.96      1115


=== Logistic Regression ===
              precision    recall  f1-score   support

      normal       0.97      1.00      0.98       966
        spam       1.00      0.80      0.89       149

    accuracy                           0.97      1115
   macro avg       0.98      0.90      0.94      1115
weighted avg       0.97      0.97      0.97      1115



In [14]:
# Réentraîner le modèle pour les tests
nb_model = MultinomialNB()
nb_model.fit(X_train_txt, y_train)


# Happy path : recall spam > 0.85
print("HAPPY PATH — Recall spam sur le jeu de test")
from sklearn.metrics import recall_score
y_pred_nb = nb_model.predict(X_test_txt)
recall_spam = recall_score(y_test, y_pred_nb)
print(f"Recall spam : {recall_spam:.2f}")
if recall_spam > 0.85:
    print("Recall > 0.85 — le modèle attrape bien les spams")
else:
    print("Recall trop bas — trop de spams passent inaperçus")
print()

# --- Edge case : message vide ---
print("EDGE CASE — Message vide")
try:
    X_vide = vect.transform([""])
    pred_vide = nb_model.predict(X_vide)
    proba_vide = nb_model.predict_proba(X_vide)
    print(f"Message vide → prédit : {'spam' if pred_vide[0] else 'normal'}")
    print(f"Probabilités : normal={proba_vide[0][0]:.2f} | spam={proba_vide[0][1]:.2f}")
    print("""Observation : le vectorizer ne plante PAS sur un message vide.
Il produit un vecteur tout à zéro → le modèle prédit la classe
majoritaire (normal) par défaut.
En production : détecter et rejeter les messages vides AVANT le modèle.
    """)
except Exception as e:
    print(f"Plantage sur message vide : {e}")


# Adversarial : spam déguisé
print("ADVERSARIAL — Spam déguisé en message normal")
spam_deguise = "salut ton colis t attend confirme ici"
X_deguise = vect.transform([spam_deguise])
pred_deguise = nb_model.predict(X_deguise)
proba_deguise = nb_model.predict_proba(X_deguise)

print(f"Message : '{spam_deguise}'")
print(f"Prédit  : {'spam ' if pred_deguise[0] else 'normal '}")
print(f"Probabilités : normal={proba_deguise[0][0]:.2f} | spam={proba_deguise[0][1]:.2f}")
print("""Observation : un spam déguisé avec des mots courants trompe facilement
le Naive Bayes car il ne connaît pas le contexte, seulement les mots.

Precision vs Recall sur le spam :
- Faux positif (normal classé spam) = on efface un vrai mail important
- Faux négatif (spam classé normal) = on laisse passer un spam

→ Sur un filtre email pro, un faux positif est souvent PIRE qu'un faux négatif.
→ Il faut regarder precision ET recall, pas juste l'accuracy.
""")

HAPPY PATH — Recall spam sur le jeu de test
Recall spam : 0.70
Recall trop bas — trop de spams passent inaperçus

EDGE CASE — Message vide
Message vide → prédit : normal
Probabilités : normal=0.87 | spam=0.13
Observation : le vectorizer ne plante PAS sur un message vide.
Il produit un vecteur tout à zéro → le modèle prédit la classe
majoritaire (normal) par défaut.
En production : détecter et rejeter les messages vides AVANT le modèle.
    
ADVERSARIAL — Spam déguisé en message normal
Message : 'salut ton colis t attend confirme ici'
Prédit  : normal 
Probabilités : normal=0.87 | spam=0.13
Observation : un spam déguisé avec des mots courants trompe facilement
le Naive Bayes car il ne connaît pas le contexte, seulement les mots.

Precision vs Recall sur le spam :
- Faux positif (normal classé spam) = on efface un vrai mail important
- Faux négatif (spam classé normal) = on laisse passer un spam

→ Sur un filtre email pro, un faux positif est souvent PIRE qu'un faux négatif.
→ Il faut re

In [23]:
# PHASE D : Sonar (mines vs rochers)

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Chargement du dataset Sonar
def charger_sonar():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/undocumented/connectionist-bench/sonar/sonar.all-data"

    df = pd.read_csv(url, header=None)

    X = df.iloc[:, :60].values
    y = (df.iloc[:, 60] == "M").astype(int).values

    print(f"Sonar : {X.shape}, mines={y.sum()}, rochers={len(y)-y.sum()}")

    return X, y

# Préparation des données
X_son, y_son = charger_sonar()

X_tr, X_te, y_tr, y_te = train_test_split(
    X_son,
    y_son,
    test_size=0.2,
    random_state=42,
    stratify=y_son
)

# Normalisation
scaler_son = StandardScaler()

X_tr_s = scaler_son.fit_transform(X_tr)
X_te_s = scaler_son.transform(X_te)

# Comparaison des modèles
modeles = [
    ("LogisticRegression", LogisticRegression(max_iter=1000)),
    ("SVC (rbf)", SVC(kernel="rbf")),
    ("RandomForest", RandomForestClassifier(n_estimators=100, random_state=42))
]

print("Résultats des modèles ")

for nom, modele in modeles:
    modele.fit(X_tr_s, y_tr)

    y_pred = modele.predict(X_te_s)

    acc = accuracy_score(y_te, y_pred)

    print(f"{nom:<22} accuracy = {acc:.2f}")


Sonar : (208, 60), mines=111, rochers=97
Résultats des modèles 
LogisticRegression     accuracy = 0.83
SVC (rbf)              accuracy = 0.93
RandomForest           accuracy = 0.81


In [22]:
from sklearn.metrics import accuracy_score

# Réentraîner les modèles proprement
svm_final = SVC(kernel="rbf")
svm_final.fit(X_tr_s, y_tr)
lr_final  = LogisticRegression(max_iter=1000)
lr_final.fit(X_tr_s, y_tr)
rf_son    = RandomForestClassifier(n_estimators=100, random_state=42)
rf_son.fit(X_tr_s, y_tr)

# Cas limite : SANS standardiser
print("CAS LIMITE — Sans standardisation")
for nom, modele in [
    ("LogisticRegression", LogisticRegression(max_iter=1000)),
    ("SVC (rbf)",          SVC(kernel="rbf")),
    ("RandomForest",       RandomForestClassifier(n_estimators=100, random_state=42)),
]:
    modele.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, modele.predict(X_te))
    print(f"{nom:<22} accuracy={acc:.2f}")
print("""
Observation :
- SVM et LogisticRegression chutent fortement sans scaling.
  Ils calculent des distances/marges → sensibles aux échelles.
- RandomForest est peu affecté : il raisonne par seuils,
  pas par distances. L'échelle ne change pas ses coupures.
""")


#Cas adversarial : capteur en panne (60 zéros)
print("ADVERSARIAL — Capteur en panne (signal tout à zéro)")
capteur_panne   = np.zeros((1, 60))
capteur_panne_s = scaler_son.transform(capteur_panne)

for nom, modele in [
    ("LogisticRegression", lr_final),
    ("SVC (rbf)",          svm_final),
    ("RandomForest",       rf_son),
]:
    pred = modele.predict(capteur_panne_s)[0]
    if hasattr(modele, "predict_proba"):
        proba = modele.predict_proba(capteur_panne_s)[0]
        confiance = max(proba)
        print(f"{nom:<22} → {'mine' if pred else 'rocher'} (confiance={confiance:.2f})")
    else:
        print(f"{nom:<22} → {'mine' if pred else 'rocher'} (SVM : pas de proba par défaut)")

print("""
Observation : les modèles prédisent quand même une classe avec assurance
alors que le signal est complètement vide (capteur en panne).
Ils ne savent PAS qu'ils ne savent pas.

En production :
1. Détecter les signaux aberrants AVANT le modèle (somme = 0 → rejet)
2. Ne jamais faire confiance à une prédiction sur une entrée hors plage
3. Ajouter un seuil de confiance minimum (ex : si max_proba < 0.7 → alarme)
→ Un vrai système embarqué a toujours une couche de validation des capteurs.
""")

CAS LIMITE — Sans standardisation
LogisticRegression     accuracy=0.81
SVC (rbf)              accuracy=0.83
RandomForest           accuracy=0.81

Observation :
- SVM et LogisticRegression chutent fortement sans scaling.
  Ils calculent des distances/marges → sensibles aux échelles.
- RandomForest est peu affecté : il raisonne par seuils,
  pas par distances. L'échelle ne change pas ses coupures.

ADVERSARIAL — Capteur en panne (signal tout à zéro)
LogisticRegression     → rocher (confiance=1.00)
SVC (rbf)              → rocher (SVM : pas de proba par défaut)
RandomForest           → rocher (confiance=0.72)

Observation : les modèles prédisent quand même une classe avec assurance
alors que le signal est complètement vide (capteur en panne).
Ils ne savent PAS qu'ils ne savent pas.

En production :
1. Détecter les signaux aberrants AVANT le modèle (somme = 0 → rejet)
2. Ne jamais faire confiance à une prédiction sur une entrée hors plage
3. Ajouter un seuil de confiance minimum (ex : si m

In [30]:
# PHASE E : Le Fight des IA — LEADERBOARD
import time
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score

def fight_des_ia(X_train, X_test, y_train, y_test, metrique, nom_metrique="Score"):
    competiteurs = {
        "LogisticRegression" : LogisticRegression(max_iter=1000),
        "DecisionTree"       : DecisionTreeClassifier(random_state=42),
        "RandomForest"       : RandomForestClassifier(n_estimators=100, random_state=42),
        "GradientBoosting"   : GradientBoostingClassifier(random_state=42),
        "SVC_rbf"            : SVC(kernel="rbf"),
    }

    resultats = []
    for nom, modele in competiteurs.items():
        debut = time.perf_counter()
        modele.fit(X_train, y_train)
        duree = time.perf_counter() - debut
        score = metrique(y_test, modele.predict(X_test))
        resultats.append((nom, score, duree))

    resultats.sort(key=lambda x: x[1], reverse=True)

    print(f"LEADERBOARD — métrique : {nom_metrique}")
    print(f"{'Rang':<5} {'Algo':<22} {nom_metrique:>8} {'Temps':>8}")
    for rang, (nom, score, duree) in enumerate(resultats, 1):
        print(f"{rang:<5} {nom:<22} {score:>8.3f} {duree:>7.2f}s")
    print(f"   Champion : {resultats[0][0]}")
    print(f"   Score : {resultats[0][1]:.3f} | Temps : {resultats[0][2]:.2f}s")


metrique_f1 = lambda y_true, y_pred: f1_score(y_true, y_pred)

fight_des_ia(X_tr_s, X_te_s, y_tr, y_te,
             metrique=metrique_f1,
             nom_metrique="F1")

LEADERBOARD — métrique : F1
Rang  Algo                         F1    Temps
1     SVC_rbf                   0.936    0.00s
2     GradientBoosting          0.864    1.16s
3     DecisionTree              0.857    0.03s
4     LogisticRegression        0.844    0.09s
5     RandomForest              0.826    0.68s
   Champion : SVC_rbf
   Score : 0.936 | Temps : 0.00s
